# 2.2 · 无状态协议与 server/discover

**Statelessness and server/discover**

> **学完本课你能：** 能够解释 2026-07-28 版 MCP 为什么是无状态的，使用服务器生成的句柄设计跨调用状态，并正确使用 server/discover 和逐请求的版本协商。

官方阅读材料：[Spec: Discovery](https://modelcontextprotocol.io/specification/2026-07-28/server/discover) · [Spec: Statelessness](https://modelcontextprotocol.io/specification/2026-07-28/basic/index#statelessness) · [Spec: Version negotiation](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 手工调用 `server/discover`
我们把一个小型商店服务器（Server）作为子进程启动，并以原始 JSON 行的形式发送 `server/discover`。服务器为发现结果声明了缓存提示，所以你能看到由服务器设置的 `ttlMs` 和 `cacheScope`。

In [ ]:
import itertools
import json
import os
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
from pathlib import Path


def show(obj, limit=1200):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text if len(text) <= limit else text[:limit] + "\n  ...")


workdir = Path(tempfile.mkdtemp())
(workdir / "shop_server.py").write_text(textwrap.dedent('''
    import json, os, secrets, sqlite3, sys, time
    from mcp.server import CacheHint, MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    INSTANCE = sys.argv[1]                       # which replica am I?
    DB = os.environ["SHOP_DB"]                   # storage shared by every replica
    mcp = MCPServer(f"shop-{INSTANCE}", version="1.0.0",
                    instructions="Create a basket with create_basket, then pass its basket_id to add_item.",
                    cache_hints={"server/discover": CacheHint(ttl_ms=3_600_000, scope="public")})

    _implicit_cart = []                          # per-process memory: the anti-pattern

    def db():
        conn = sqlite3.connect(DB)
        conn.execute("create table if not exists baskets (id text primary key, items text, expires real)")
        return conn

    @mcp.tool()
    def add_to_my_cart(sku: str) -> str:
        """ANTI-PATTERN: add to 'the' cart remembered by this process."""
        _implicit_cart.append(sku)
        return json.dumps({"served_by": INSTANCE, "cart": _implicit_cart})

    @mcp.tool()
    def create_basket() -> str:
        """Create an empty basket and return its basket_id. Baskets expire after 10 minutes of inactivity."""
        basket_id = "bsk_" + secrets.token_urlsafe(12)      # opaque, high entropy
        with db() as conn:
            conn.execute("insert into baskets values (?, ?, ?)", (basket_id, "[]", time.time() + 600))
        return json.dumps({"served_by": INSTANCE, "basket_id": basket_id})

    @mcp.tool()
    def add_item(basket_id: str, sku: str) -> str:
        """Add a product to a basket created by create_basket."""
        with db() as conn:
            row = conn.execute("select items, expires from baskets where id = ?", (basket_id,)).fetchone()
            if row is None or row[1] < time.time():
                raise ToolError(f"Unknown or expired basket {basket_id!r}. Call create_basket to start a new one.")
            items = json.loads(row[0]) + [sku]
            conn.execute("update baskets set items = ?, expires = ? where id = ?",
                         (json.dumps(items), time.time() + 600, basket_id))
        return json.dumps({"served_by": INSTANCE, "basket_id": basket_id, "items": items})

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

ENV = {**os.environ, "SHOP_DB": str(workdir / "shop.sqlite")}
ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "0.1"}}


class Replica:
    """One server process we talk to with raw JSON lines."""

    def __init__(self, name):
        self.name = name
        self.proc = subprocess.Popen([sys.executable, str(workdir / "shop_server.py"), name], env=ENV,
                                     stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                                     text=True)
        self.inbox = queue.Queue()
        threading.Thread(target=lambda: [self.inbox.put(l) for l in self.proc.stdout], daemon=True).start()

    def send(self, message: dict, wait=10):
        self.proc.stdin.write(json.dumps(message) + "\n")
        self.proc.stdin.flush()
        return json.loads(self.inbox.get(timeout=wait))

    def rpc(self, id_, method, params=None, meta=ENVELOPE):
        return self.send({"jsonrpc": "2.0", "id": id_, "method": method, "params": {**(params or {}), "_meta": meta}})

    def close(self):
        self.proc.stdin.close()
        return self.proc.wait(timeout=10)


a = Replica("A")
discover = a.rpc("discover-1", "server/discover")
show(discover)

像宿主（Host）那样把 `DiscoverResult` 拆开来看：

In [ ]:
d = discover["result"]
print("server          :", d["_meta"]["io.modelcontextprotocol/serverInfo"])
print("versions        :", d["supportedVersions"])
print("capabilities    :", sorted(d["capabilities"]))
print("instructions    :", d.get("instructions"))
print(f"cache           : fresh for {d['ttlMs'] / 1000:.0f}s, scope={d['cacheScope']!r} "
      "(public = a shared gateway may cache it)")

### 2. 逐请求的版本协商
没有握手：我们直接以一个服务器不认识的版本调用工具，读取 `-32022`，从 `data.supported` 中选出一个双方都支持的版本，然后重试。这就是规范所描述的客户端版本协商（Version Negotiation）逻辑，大约十行代码。

In [ ]:
CLIENT_VERSIONS = ["2099-01-01", "2026-07-28"]          # newest first


def call_with_negotiation(replica, method, params, preferred=CLIENT_VERSIONS[0]):
    version, attempt = preferred, 0
    while True:
        attempt += 1
        reply = replica.rpc(100 + attempt, method, params,
                            meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": version})
        if "error" in reply and reply["error"]["code"] == -32022:
            supported = reply["error"]["data"]["supported"]
            print(f"attempt {attempt}: {version} rejected; server supports {supported}")
            mutual = [v for v in CLIENT_VERSIONS if v in supported]
            if not mutual:
                raise RuntimeError("no mutually supported protocol version")
            version = mutual[0]
            continue
        print(f"attempt {attempt}: accepted at {version}")
        return reply


reply = call_with_negotiation(a, "tools/call", {"name": "create_basket", "arguments": {}})
print(reply["result"]["content"][0]["text"])

### 3. 任何副本都能作答：两个独立的进程
启动第二个进程 `B`。除了<em>应用</em>选择共享的数据库文件之外，它与 `A` 没有任何共享。一个轮询“负载均衡器”（Round-robin Load Balancing）把每个请求发给下一个副本。

In [ ]:
b = Replica("B")
balancer = itertools.cycle([a, b])
ids = itertools.count(1000)


def lb_call(tool, arguments):
    replica = next(balancer)
    reply = replica.rpc(next(ids), "tools/call", {"name": tool, "arguments": arguments})
    result = reply["result"]
    return replica.name, result["isError"], result["content"][0]["text"]


for _ in range(2):
    replica = next(balancer)
    info = replica.rpc(next(ids), "server/discover")["result"]["_meta"]["io.modelcontextprotocol/serverInfo"]
    print(f"discover via {replica.name}: {info}")

### 4. 把它弄坏：隐式的进程内状态
`add_to_my_cart` 把“那个”购物车保存在进程内存中，就像会话时代的服务器可能把它保存在会话（Session）里一样。在负载均衡器后面，用户的购物车被拆散到了不同的副本上：

In [ ]:
for sku in ["apple", "pear", "fig", "kiwi"]:
    name, is_error, text = lb_call("add_to_my_cart", {"sku": sku})
    print(f"via {name}: {text}")

没有哪个副本拥有完整的购物车，而协议中也没有任何东西能修复这个问题。服务器从进程推断了连续性，而这正是规范所禁止的。

### 5. 修复它：作为工具参数传递的服务器生成句柄
现在按照规范推荐的方式实现这个有状态的工作流。`create_basket` 生成一个不透明的 id，之后的每次调用都指明它。无论哪个副本处理调用，都能在共享存储中找到这个购物篮。这就是服务器生成的句柄（Server-minted Handle）。

In [ ]:
name, _, text = lb_call("create_basket", {})
basket_id = json.loads(text)["basket_id"]
print(f"created by {name}: {basket_id}")

for sku in ["apple", "pear", "fig", "kiwi"]:
    name, is_error, text = lb_call("add_item", {"basket_id": basket_id, "sku": sku})
    print(f"via {name}: {json.loads(text)['items']}")

name, is_error, text = lb_call("add_item", {"basket_id": "bsk_does_not_exist", "sku": "x"})
print(f"\nunknown handle via {name}: isError={is_error} -> {text}")

未知句柄的情况是一个<strong>工具执行错误（Tool Execution Error）</strong>（`isError: true`），其消息告诉模型如何恢复，这正是规范中“过期错误”指导所要求的。（在这个 SDK 中，抛出 `ToolError` 可以发送你自己的消息。任何其他异常都会被报告为一条通用的“Error executing tool …”。）

### 6. 能力按请求逐一判断，即使在同一个连接上
在<strong>同一个</strong>进程上发两个请求，一个声明了 `elicitation`，另一个没有。服务器不得把第一次的声明沿用到第二次。这就是能力（Capability）的逐请求判断。

In [ ]:
(workdir / "confirm_server.py").write_text(textwrap.dedent('''
    from typing import Annotated
    from pydantic import BaseModel
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Elicit, Resolve

    mcp = MCPServer("confirm")

    class Confirm(BaseModel):
        ok: bool

    def ask(basket_id: str):
        return Elicit(f"Empty basket {basket_id}?", Confirm)

    @mcp.tool()
    def empty_basket(basket_id: str, confirm: Annotated[Confirm, Resolve(ask)]) -> str:
        """Empty a basket after the user confirms."""
        return "emptied" if confirm.ok else "kept"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
confirm = Replica.__new__(Replica)
confirm.name = "confirm"
confirm.proc = subprocess.Popen([sys.executable, str(workdir / "confirm_server.py")], stdin=subprocess.PIPE,
                                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
confirm.inbox = queue.Queue()
threading.Thread(target=lambda: [confirm.inbox.put(l) for l in confirm.proc.stdout], daemon=True).start()

args = {"name": "empty_basket", "arguments": {"basket_id": basket_id}}
with_cap = confirm.rpc(1, "tools/call", args, meta={**ENVELOPE, "io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}})
without = confirm.rpc(2, "tools/call", args)
print("request 1 (declares elicitation):", with_cap["result"]["resultType"])
print("request 2 (declares nothing)    :", without["error"]["code"], without["error"]["message"][:60])
print("                                  ", without["error"]["data"])

### 7. 旧时代：在新进程上调用 `initialize`
这个 SDK 是<strong>双时代（Dual-era）</strong>的。如果一个 stdio 进程收到的第一个请求是 2025 风格的 `initialize`，整个进程都会以旧版语义提供服务，在它上面的新版请求会被拒绝。这就是为什么双时代的<em>客户端</em>要先用 `server/discover` 探测。

In [ ]:
old = Replica("legacy")
show(old.send({"jsonrpc": "2.0", "id": 1, "method": "initialize",
               "params": {"protocolVersion": "2025-11-25", "capabilities": {},
                          "clientInfo": {"name": "old-client", "version": "1.0"}}}), limit=600)
old.proc.stdin.write(json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}) + "\n")
old.proc.stdin.flush()
show(old.rpc(2, "server/discover"))

### 8. SDK 会替你完成这一切
`Client` 在默认的 `mode="auto"` 下会探测 `server/discover`，采用其结果，并在之后的每个请求上加盖信封（Per-request Envelope）。`mode="legacy"` 则强制使用旧的握手。对比一下：

In [ ]:
from mcp import Client, StdioServerParameters

params = StdioServerParameters(command=sys.executable, args=[str(workdir / "shop_server.py"), "C"], env=ENV)
async with Client(params) as client:
    d = client.session.discover_result
    print("auto  :", client.protocol_version, "| discover:", d.supported_versions, d.ttl_ms, d.cache_scope)
    print("        instructions:", client.instructions)
async with Client(params, mode="legacy") as client:
    print("legacy:", client.protocol_version, "| discover result:", client.session.discover_result)

for replica in (a, b, confirm, old):
    replica.close()

### 动手练习
1. 添加一个 `view_basket(basket_id)` 工具（Tool），并确认通过副本 `A` 创建的购物篮可以通过副本 `B` 查看。然后在两次调用之间删除 `shop.sqlite`：这个工具应该返回什么？你的实现是这样的吗？
2. 让 `create_basket` 生成 `bsk_1`、`bsk_2`、…… 这样的 id，并结合规范中关于句柄的四个关注点，解释这在无需认证的服务器上会引发的两种具体攻击。
3. 把 discover 的缓存提示改为 `scope="private"`。谁仍然可以缓存这个结果？什么情况下你会需要对发现结果使用 `private`？
4. 编写 `probe_era(replica)`，实现本课中的 stdio 向后兼容规则（discover → 新版；可识别的新版错误 → 新版；其他任何情况或超时 → 旧版）。在一个新副本上测试它，再在另一个先按第 7 节那样用 `initialize` 打开的新副本上测试它。